<a href="https://colab.research.google.com/github/luke-wellerman/luke-wellerman-ds2002-fa26/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one: An ordered Item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
print('checks passed.')

checks passed.


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors_df = pd.json_normalize(vendors_json)
joined = lines.merge(vendors_df, on='vendor_id', how='left', indicator=True)
unmatched = joined[joined['_merge'] == 'left_only']
print(unmatched)

      item  qty order vendor_id name zone     _merge
3  Hot Dog    3     3      V-07  NaN  NaN  left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
vendor_totals = joined.groupby('name', dropna=False)['qty'].sum()
print(vendor_totals)

name
Cav Merch       1
Hoos Burgers    3
NaN             3
Name: qty, dtype: int64


The unmatched vendor has a NaN name because of left join.
This method keeps NaN in a seperate group rather than dropping it.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
zoneTotal = joined.groupby('zone', dropna=False)['qty'].sum()
print(zoneTotal)


zone
A      4
NaN    3
Name: qty, dtype: int64


Zone A sold the most (4) items. Total across known = 4, with a difference of 3 units since vender 07 was missing from vendor directory. Total units = 3+4=7

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
cleaned_ragged = [
    {**record, 'lines': record.get('lines', [])}
    for record in ragged_json
]
ragged_df = pd.json_normalize(
    cleaned_ragged,
    record_path='lines',
    meta=['order', 'vendor_id']
)
print(ragged_df)
print()
print("Qty Null:")
print(ragged_df[['order', 'item', 'qty']].isna())

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01

Qty Null:
   order   item    qty
0  False  False  False
1  False  False   True


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A) One row per line item ensures you can sum quantities and filter by item type. If each row was an order, more work would be required to determine the same answers.

B) The quantity of 0 means that the value is known while NA/null means the value was unknown. The 0 could mean a cancelled order. Conflating them will skew statistics and incorrect data by making unknown values a set value.